[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/articulated/articulated.ipynb)

# Articulated Robot Dynamics

This example simulates the reduced-coordinate dynamics of a branching humanoid robot falling onto level ground. A floating root torso carries a tree of limbs turning on passive hinge joints with one-sided range stops, while capsules and spheres mounted on the links collide with the ground under damped spring pushes and dry sliding friction. Inverse dynamics finds the torso load and joint generalized forces required for given accelerations, forward dynamics solves the coupled mass matrix for accelerations under applied loads, and a linearly implicit Euler integrator steps the limp humanoid as it lands on its feet, buckles at the knees, and slumps into a heap.

In projective 3D geometric algebra, each hinge axis and link spatial velocity is modelled as an extrinsic line of turn, while each link momentum and applied forque is modelled as an intrinsic line joined from points ([Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)). One motor sandwich carries motions, forces, and spatial ***duality map*** inertias `motor >> inertia(motor << Rate)` between frames, and one commutator evaluates both rigid-body cross products: `rate.commutator(axis)` turns a hinge axis carried by a moving link, while `rate.commutator(momentum)` transports momentum under motion with orientation keeping motions and forces distinct. Summing hinge speeds along each root-to-link path forms `joint_rates`, whose adjugate pulls subtree forques back onto each hinge via `axis & forque` and pulls link inertias back into the tree-sparse joint mass matrix `joint_rates.adjugate()(inertias(joint_rates))`. Over an implicit step, ground damping and stiffness joined at each ball's lowest point enter as extra link rate-to-forque maps pulled back through that same operator.

In Featherstone's spatial-vector notation, six-component motion and force vectors transform under distinct $6 \times 6$ matrices ${}^B X_A$ and ${}^B X_A^* = {}^B X_A^{-\mathsf{T}}$ so spatial inertias transform by congruence $I_B = {}^B X_A^* I_A {}^A X_B$, while motions and forces require separate cross-product operators $\operatorname{crm}(v) = v \times$ and $\operatorname{crf}(v) = v \times^* = -\operatorname{crm}(v)^{\mathsf{T}}$. Recursive Newton–Euler sweeps $v_i = v_{\lambda(i)} + S_i \dot{q}_i$ and $a_i = a_{\lambda(i)} + S_i \ddot{q}_i + v_i \times S_i \dot{q}_i$ outward from the root, evaluates $f_i = I_i a_i + v_i \times^* I_i v_i - f_i^x$, and sweeps $\tau_i = S_i^{\mathsf{T}} f_i$ and $f_{\lambda(i)} \mathrel{+}= f_i$ inward before assembling $H_{ij} = S_i^{\mathsf{T}} I_i^C S_j$ from subtree composite inertias $I_i^C$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[jax,examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import jax
import numpy as np

from numga import Extensor, stack, where
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology
from examples.mechanics.articulated import render, scenarios
from examples.mechanics.spinning_top import core as contact
from examples.util.animation import show_states

jax.config.update("jax_enable_x64", True)
mv = JaxContext(ga, dtype=np.float64).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Point = ga.gatype.antivector()                                               # Antivector-
Plane = ga.gatype.vector()                                                   # Vector+
# Joint axis per unit speed and link spatial velocity: an extrinsic line of turn.
Rate = ga.gatype.bivector()                                                  # Bivector+
# Link spatial momentum and applied forque: an intrinsic line joined from points.
Forque = ga.gatype.antibivector()                                            # Antibivector-
Motor = ga.gatype.rotor()                                                    # Rotor+
# Generalized joint momentum and force paired against scalar joint speed.
Force = ga.gatype.antiscalar()                                               # Antiscalar-
Inertia = ga.gatype((Forque, Rate))                                          # Forque- <- Rate+
Hinge = ga.gatype((Force, Scalar))                                           # Force- <- Scalar+
Quadric = ga.gatype((Plane, Point))                                          # Plane+ <- Point-

Tree = tuple[Sparse, Sparse, Sparse, Sparse, Sparse, int]
State = tuple[Motor, Rate, Scalar, Scalar]

## Kinematic tree and forward kinematics

The humanoid forms a directed tree of links connected by single-axis hinges rooted at the torso, with massless intermediate links splitting multi-axis waist, hip, and shoulder joints into chains of single hinges. Each solid capsule and sphere carries six principal mass samples, so `(masses * samples).regressive(samples.commutator(Rate))` sums into each part's spatial inertia map from rates to momenta and pulls back onto the carrying links, while the capsule endpoints and spheres define the balls that can touch the ground. Forward kinematics places the root torso by its motor and propagates each child link's placement across the tree levels by composing its parent's motor with its home-pose hinge turn.

In [ ]:
def kinematic_tree(topology: Topology, ball_carriers: np.ndarray, n_generations: int) -> Tree:
    """Root reading (`Scalar+[1 <- V]`), edge tail and head readings (`Scalar+[E <- V]`), root-to-link path matrix (`Scalar+[V <- E]`), ball-carrier reading (`Scalar+[n_balls <- V]`), and tree depth `n_generations` assemble from the directed joint `topology`."""
    V, E = topology.n_elements
    # Read each directed joint's parent tail and child head separately so every sparse map stores only active couplings.
    tails, heads = (topology.reading(ends, 0) for ends in topology.elements[1].T)
    parents = heads.adjugate() * tails                                       # Scalar+[V <- V]
    paths = heads.adjugate()                                                 # Scalar+[V <- E]
    for _ in range(n_generations - 1):
        paths = heads.adjugate() + parents * paths
    root = np.setdiff1d(np.arange(V), topology.elements[1][:, 1])
    return topology.reading(root, 0), tails, heads, paths, topology.reading(ball_carriers, 0), n_generations


@partial(jax.jit, static_argnames=("tree",))
def forward_kinematics(tree: Tree, home_axes: Rate, motor: Motor, angles: Scalar) -> Motor:
    """Link placement motors (`[...] Motor+[V]`) compose the root `motor` with each joint's home-pose turn across the `n_generations` levels of `tree`."""
    root, tails, heads, _, _, n_generations = tree
    turns = (home_axes * (angles / 2)).exp()                                 # [...] Motor+[E]
    root_placement = root.adjugate() * motor[..., None].field()              # [...] Motor+[V]
    motors = root_placement                                                  # [...] Motor+[V]
    for _ in range(n_generations):
        motors = root_placement + heads.adjugate() * ((tails * motors) * turns)
    return motors


topology, carriers, _, n_generations = scenarios.humanoid()
n_capsules = len(scenarios.capsule_bodies)
ball_carriers = carriers[np.concatenate([scenarios.capsule_bodies, scenarios.capsule_bodies, scenarios.sphere_bodies])]
part_carriers = carriers[np.concatenate([scenarios.capsule_bodies, scenarios.sphere_bodies])]
tree = kinematic_tree(topology, ball_carriers, n_generations)
part_reading = topology.reading(part_carriers, 0)                            # Scalar+[n_parts <- V]
home_axes, samples, masses, centres, radii = scenarios.home_pose()
n_parts, n_balls = len(samples), len(radii)
part_inertias = (masses * samples).regressive(samples.commutator(Rate)).sum(axis=-1)   # [n_parts] Forque- <- Rate+
inertias = part_reading.adjugate() * part_inertias.field()                             # (Forque- <- Rate+)[V]
mass_points = part_reading.adjugate() * (masses * samples).sum(axis=-1).field()        # Point-[V]

initial_state = scenarios.initial_drop_state()
initial_balls = (tree[4] * forward_kinematics(tree, home_axes, initial_state[0], initial_state[2])).batch()   # [1, n_balls] Motor+
render.pose((initial_balls >> centres)[0], radii, n_capsules);

## Inverse dynamics

Inverse dynamics balances link inertia and momentum transport at specified velocities and accelerations against external forques. Carrying each home-pose hinge axis by its parent link's motor and summing along the root-to-link paths builds `joint_rates`, which maps hinge speeds onto link spatial velocities on top of the root torso's rate. Because each moving link turns the hinge axis it carries, child link accelerations pick up `rate.commutator(axis) * speed` down each subtree, while each link's momentum turns at `rate.commutator(momentum)`. Summing the resulting net link forques gives the root load, and `joint_rates.adjugate()` pulls subtree forques back onto each hinge via `axis & forque`.

In [ ]:
def inverse_dynamics(
    tree: Tree,
    home_axes: Rate,
    inertias: Inertia,
    motors: Motor,
    rate: Rate,
    speeds: Scalar,
    acceleration: Rate,
    accelerations: Scalar,
    external_forques: Forque,
) -> tuple[Forque, Force]:
    """Root forque (`[...] Forque-`) and joint generalized forces (`[...] Force-[E]`) balance link inertia and momentum transport at `(rate, speeds)` and `(acceleration, accelerations)` against `external_forques`."""
    _, tails, heads, paths, _, _ = tree
    axes = (tails * motors) >> home_axes                                     # [...] Rate+[E]
    joint_rates = paths * spdiag(axes * Scalar)                              # [...] (Rate+ <- Scalar+)[V <- E]
    world_inertias = motors >> inertias(motors << Rate)                      # [...] (Forque- <- Rate+)[V]
    # Propagate link rates outward; each turning child link adds `rate.commutator(axis) * speed` down its subtree.
    rates = joint_rates(speeds) + rate                                       # [...] Rate+[V]
    velocity_products = (heads * rates).commutator(axes) * speeds            # [...] Rate+[E]
    link_accelerations = joint_rates(accelerations) + paths * velocity_products + acceleration   # [...] Rate+[V]
    momenta = world_inertias(rates)                                          # [...] Forque-[V]
    forques = world_inertias(link_accelerations) + rates.commutator(momenta) - external_forques  # [...] Forque-[V]
    # Sum all link forques onto the root and pull subtree forques back onto each joint axis.
    return forques.sites.sum(), joint_rates.adjugate()(forques)

## Mass matrix and forward dynamics

Pulling any link rate-to-forque map back through the root and hinge kinematics splits the coupled system into three blocks: the whole-robot inertia summed onto the root, the root-to-hinge coupling, and the symmetric hinge block `joint_rates.adjugate()(link_maps(joint_rates))`, which couples two hinges only when one lies on the path to the other. Eliminating the six-degree-of-freedom root block leaves a compact hinge system per robot, and forward dynamics solves those blocks against applied loads minus the zero-acceleration bias loads from inverse dynamics.

In [ ]:
def rate_pullback(joint_rates: SparseExtensor, link_maps: Inertia) -> tuple[Inertia, Extensor, SparseExtensor]:
    """Link rate-to-forque maps (`[...] (Forque- <- Rate+)[V]`) pull back into the root block (`[...] Forque- <- Rate+`), root-joint coupling (`[...] (Force- <- Rate+)[E]`), and sparse joint block (`[...] (Force- <- Scalar+)[E <- E]`)."""
    return (
        link_maps.sites.sum(),
        joint_rates.adjugate()(link_maps),
        joint_rates.adjugate()(link_maps(joint_rates)),
    )


def coupled_rates(
    root_block: Inertia,
    coupling: Extensor,
    joint_block: SparseExtensor,
    forque: Forque,
    forces: Force,
) -> tuple[Rate, Scalar]:
    """Root rate (`[...] Rate+`) and joint speeds (`[...] Scalar+[E]`) solve the coupled block system against `(forque, forces)` by Schur elimination of `root_block`."""
    # Eliminate the six-degree-of-freedom root block and solve each robot's small joint system densely on JAX.
    reduced = joint_block.to_dense() - coupling(root_block.solve(coupling.adjugate()))   # [...] (Force- <- Scalar+)[E <- E]
    speeds = reduced.solve(forces - coupling(root_block.solve(forque)))                  # [...] Scalar+[E]
    return root_block.solve(forque - coupling.adjugate()(speeds)), speeds

In [ ]:
def mass_matrix(tree: Tree, home_axes: Rate, inertias: Inertia, motors: Motor) -> tuple[Inertia, Extensor, SparseExtensor]:
    """World-frame link inertias pull back at `motors` into the root block, root-joint coupling, and sparse joint block of the mass matrix."""
    _, tails, _, paths, _, _ = tree
    axes = (tails * motors) >> home_axes                                     # [...] Rate+[E]
    joint_rates = paths * spdiag(axes * Scalar)                              # [...] (Rate+ <- Scalar+)[V <- E]
    return rate_pullback(joint_rates, motors >> inertias(motors << Rate))


def forward_dynamics(
    tree: Tree,
    home_axes: Rate,
    inertias: Inertia,
    motors: Motor,
    rate: Rate,
    speeds: Scalar,
    forces: Force,
    external_forques: Forque,
) -> tuple[Rate, Scalar]:
    """Root angular-linear acceleration (`[...] Rate+`) and joint accelerations (`[...] Scalar+[E]`) solve the mass matrix against `(forces, external_forques)` minus zero-acceleration bias loads."""
    root_bias, joint_bias = inverse_dynamics(
        tree, home_axes, inertias, motors, rate, speeds, rate * 0, speeds * 0, external_forques
    )
    return coupled_rates(*mass_matrix(tree, home_axes, inertias, motors), -root_bias, forces - joint_bias)

## Gravity and ground contact

Downward gravity joins with each link's weighted mass point as `gravity & (motors >> mass_points)` to produce its weight forque. Where a contact ball penetrates the ground plane at its lowest point, a damped normal spring pushes upward without pulling on rebound while Coulomb damping resists tangential slip in proportion to the normal push. Joining those contact damping and stiffness operators with each ball's lowest point via `bottoms.regressive(dampers(bottoms.commutator(Rate)))` gives link rate-to-forque maps of the same type as spatial inertia.

In [ ]:
gravity = -mv.z.dual() * scenarios.gravity_strength                          # [] Point-
ground, surfaces, contact_stiffness, contact_damping, friction = scenarios.ground_parameters()


def ground_contact(
    balls: Sparse,
    ground: Plane,
    surfaces: Quadric,
    contact_stiffness: Scalar,
    contact_damping: float,
    friction: float,
    motors: Motor,
    rates: Rate,
) -> tuple[Forque, Inertia, Inertia]:
    """Ground contact at each ball's lowest point produces link contact forques (`[...] Forque-[V]`), contact damping maps (`[...] (Forque- <- Rate+)[V]`), and contact stiffness maps (`[...] (Forque- <- Rate+)[V]`)."""
    up_direction = ground.dual()                                             # [] Point-
    ball_motors = (balls * motors).batch()                                   # [...] Motor+
    world_surfaces = ball_motors >> surfaces(ball_motors << Point)           # [...] Plane+ <- Point-
    bottoms = contact.lowest_point(world_surfaces, up_direction).field()     # [...] Point-[n_balls]
    velocities = (balls * rates).commutator(bottoms)                         # [...] Point-[n_balls]
    normal_part = up_direction * (ground & Point)                            # [] Point- <- Point-
    depths = (-(ground & bottoms)).clip(0, np.inf)                           # [...] Scalar+[n_balls]
    sinking_speeds = -(ground & velocities)                                  # [...] Scalar+[n_balls]
    # Clamp normal forces nonnegative so rising contacts never pull a ball toward the ground.
    pushes = (depths * contact_stiffness * (1 + contact_damping * sinking_speeds)).clip(0, np.inf)   # [...] Scalar+[n_balls]
    forces = up_direction * pushes - (velocities - normal_part(velocities)) * (pushes * friction)    # [...] Point-[n_balls]
    dampers = normal_part * (depths * contact_stiffness * contact_damping) + (Point - normal_part) * (pushes * friction)
    springs = normal_part * where(depths > 0, contact_stiffness, 0)          # [...] (Point- <- Point-)[n_balls]
    damping_maps = bottoms.regressive(dampers(bottoms.commutator(Rate)))     # [...] (Forque- <- Rate+)[n_balls]
    stiffness_maps = bottoms.regressive(springs(bottoms.commutator(Rate)))   # [...] (Forque- <- Rate+)[n_balls]
    return balls.adjugate() * (forces & bottoms), balls.adjugate() * damping_maps, balls.adjugate() * stiffness_maps

## Linearly implicit stepping and fall onto level ground

Weak passive hinge springs and dampers draw the limbs gently toward the home pose while stiff one-sided range stops prevent the knees and elbows from hyperextending. A linearly implicit Euler step evaluates hinge, stop, and ground spring loads at the end-of-step positions predicted from current velocities and adds damping times step size plus stiffness times step size squared directly to the coupled system: hinge and stop terms enter the hinge diagonal, while ground contact maps add to the link inertias before `rate_pullback`. Compiling the substep loop with JAX streams the sixty-frame fall in a few seconds.

In matrix notation, the linearly implicit step solves $(M + h D + h^2 K)\,\Delta v = h\,(f - h K v)$ over time step $h$, where ground damping $D_c$ and stiffness $K_c$ enter $D$ and $K$ through congruence $J_c^{\mathsf{T}} D_c J_c$ and $J_c^{\mathsf{T}} K_c J_c$ with the contact-point Jacobian $J_c$.

In [ ]:
stiffness, damping = scenarios.hinge_parameters()
hinge_stiffness = stiffness * Scalar.dual()                                  # (Force- <- Scalar+)[E]
hinge_damping = damping * Scalar.dual()                                      # (Force- <- Scalar+)[E]
low_stops, high_stops, stop_raw = scenarios.stop_parameters()
stop_stiffness = stop_raw * Scalar.dual()                                    # (Force- <- Scalar+)[E]
dt, n_steps, n_frames, frame_duration_ms = scenarios.dt, scenarios.n_steps, scenarios.n_frames, scenarios.frame_duration_ms


def free_step(
    tree: Tree,
    home_axes: Rate,
    inertias: Inertia,
    mass_points: Point,
    gravity: Point,
    hinge_stiffness: Hinge,
    hinge_damping: Hinge,
    state: State,
    dt: float,
) -> State:
    """Linearly implicit Euler advances `state` by `dt` in free flight under gravity and passive hinge springs and dampers."""
    _, tails, _, paths, _, _ = tree
    motor, rate, angles, speeds = state
    motors = forward_kinematics(tree, home_axes, motor, angles)              # [...] Motor+[V]
    axes = (tails * motors) >> home_axes                                     # [...] Rate+[E]
    joint_rates = paths * spdiag(axes * Scalar)                              # [...] (Rate+ <- Scalar+)[V <- E]
    weights = gravity & (motors >> mass_points)                              # [...] Forque-[V]
    drives = -(hinge_stiffness(angles + speeds * dt) + hinge_damping(speeds))   # [...] Force-[E]
    root_bias, joint_bias = inverse_dynamics(
        tree, home_axes, inertias, motors, rate, speeds, rate * 0, speeds * 0, weights
    )
    root_block, coupling, joint_block = rate_pullback(joint_rates, motors >> inertias(motors << Rate))
    hinge_maps = hinge_damping * dt + hinge_stiffness * dt**2                # [...] (Force- <- Scalar+)[E]
    rate_change, speed_changes = coupled_rates(
        root_block, coupling, joint_block + hinge_maps, -root_bias * dt, (drives - joint_bias) * dt
    )
    next_rate, next_speeds = rate + rate_change, speeds + speed_changes
    next_turn = (next_rate * (dt / 2)).exp()                                 # [...] Motor+
    return next_turn * motor, next_rate, angles + next_speeds * dt, next_speeds


def drop_step(
    tree: Tree,
    home_axes: Rate,
    inertias: Inertia,
    mass_points: Point,
    gravity: Point,
    hinge_stiffness: Hinge,
    hinge_damping: Hinge,
    low_stops: Scalar,
    high_stops: Scalar,
    stop_stiffness: Hinge,
    ground: Plane,
    surfaces: Quadric,
    contact_stiffness: Scalar,
    contact_damping: float,
    friction: float,
    state: State,
    dt: float,
) -> State:
    """Linearly implicit Euler advances `state` by `dt` under gravity, hinge springs and dampers, one-sided joint stops, and ground contact."""
    _, tails, _, paths, balls, _ = tree
    motor, rate, angles, speeds = state
    motors = forward_kinematics(tree, home_axes, motor, angles)              # [...] Motor+[V]
    axes = (tails * motors) >> home_axes                                     # [...] Rate+[E]
    joint_rates = paths * spdiag(axes * Scalar)                              # [...] (Rate+ <- Scalar+)[V <- E]
    rates = joint_rates(speeds) + rate                                       # [...] Rate+[V]
    pushes, contact_damping_maps, contact_stiffness_maps = ground_contact(
        balls, ground, surfaces, contact_stiffness, contact_damping, friction, motors, rates
    )
    weights = gravity & (motors >> mass_points)                              # [...] Forque-[V]
    # Shift ground spring forces to end-of-step positions predicted from current link rates.
    external_forques = weights + pushes - contact_stiffness_maps(rates) * dt   # [...] Forque-[V]
    overruns = (angles - high_stops).clip(0, np.inf) + (angles - low_stops).clip(-np.inf, 0)   # [...] Scalar+[E]
    active_stops = where(overruns.abs() > 0, stop_stiffness, 0)              # [...] (Force- <- Scalar+)[E]
    angle_steps = speeds * dt                                                # [...] Scalar+[E]
    drives = -(hinge_stiffness(angles + angle_steps) + active_stops(overruns + angle_steps) + hinge_damping(speeds))
    root_bias, joint_bias = inverse_dynamics(
        tree, home_axes, inertias, motors, rate, speeds, rate * 0, speeds * 0, external_forques
    )
    link_maps = (motors >> inertias(motors << Rate)) + contact_damping_maps * dt + contact_stiffness_maps * dt**2
    root_block, coupling, joint_block = rate_pullback(joint_rates, link_maps)
    hinge_maps = hinge_damping * dt + (hinge_stiffness + active_stops) * dt**2   # [...] (Force- <- Scalar+)[E]
    rate_change, speed_changes = coupled_rates(
        root_block, coupling, joint_block + hinge_maps, -root_bias * dt, (drives - joint_bias) * dt
    )
    next_rate, next_speeds = rate + rate_change, speeds + speed_changes
    next_turn = (next_rate * (dt / 2)).exp()                                 # [...] Motor+
    return next_turn * motor, next_rate, angles + next_speeds * dt, next_speeds

In [ ]:
@partial(jax.jit, static_argnames=("step_fn", "n_steps"))
def advance(step_fn, state: State, dt: float, n_steps: int) -> State:
    """Repeated `step_fn` application advances `state` across `n_steps` substeps of `dt` inside a single compiled loop."""
    joint_type = state[2].gatype

    def retyped_state(current: State) -> State:
        """Full carry types stay invariant across `fori_loop` iterations."""
        motor, rate, angles, speeds = current
        return motor.retyped(Motor), rate.retyped(Rate), angles.retyped(joint_type), speeds.retyped(joint_type)

    return jax.lax.fori_loop(0, n_steps, lambda _, current: retyped_state(step_fn(current, dt)), retyped_state(state))


def trajectory(step_fn, state: State, dt: float, n_steps: int, n_frames: int) -> Iterator[State]:
    """Yields `n_frames` robot states (`State`) spaced by `n_steps` substeps of `dt` starting from `state`."""
    for _ in range(n_frames):
        yield state
        state = advance(step_fn, state, dt, n_steps)


step_fn = partial(
    drop_step,
    tree,
    home_axes,
    inertias,
    mass_points,
    gravity,
    hinge_stiffness,
    hinge_damping,
    low_stops,
    high_stops,
    stop_stiffness,
    ground,
    surfaces,
    contact_stiffness,
    contact_damping,
    friction,
)
forward_fn = partial(forward_kinematics, tree, home_axes)
states = show_states(
    trajectory(step_fn, initial_state, dt, n_steps, n_frames),
    partial(render.animate_states, forward_fn, tree[4], centres, radii, n_capsules),
    name="articulated",
    duration_ms=frame_duration_ms,
)

Dropped tilted and spinning above the slab, the humanoid touches down feet first, folds at the ankles, knees, and hips as ground friction catches its feet, and settles onto the floor without rebounding; self-collision between links is omitted so only the ground slab supports the limbs.

## Mechanical energy

Total mechanical energy sums the coupled kinetic energy of the root and hinges with gravitational potential energy and the elastic energy stored in the passive hinges, exceeded range stops, and compressed ground contacts.

In [ ]:
@partial(jax.jit, static_argnames=("tree",))
def mechanical_energy(
    tree: Tree,
    home_axes: Rate,
    inertias: Inertia,
    mass_points: Point,
    gravity: Point,
    hinge_stiffness: Hinge,
    low_stops: Scalar,
    high_stops: Scalar,
    stop_stiffness: Hinge,
    ground: Plane,
    surfaces: Quadric,
    contact_stiffness: Scalar,
    state: State,
) -> Scalar:
    """Total mechanical energy (`[...] Scalar+`) sums kinetic energy from the mass matrix blocks with gravitational, hinge-spring, joint-stop, and ground-spring potential energies."""
    _, _, _, _, balls, _ = tree
    motor, rate, angles, speeds = state
    motors = forward_kinematics(tree, home_axes, motor, angles)              # [...] Motor+[V]
    root_block, coupling, joint_block = mass_matrix(tree, home_axes, inertias, motors)
    kinetic = (
        (rate & root_block(rate)) / 2
        + (speeds & coupling(rate)).sites.sum()
        + (speeds & joint_block(speeds)).sites.sum() / 2
    )
    overruns = (angles - high_stops).clip(0, np.inf) + (angles - low_stops).clip(-np.inf, 0)   # [...] Scalar+[E]
    spring_energy = (
        (angles & hinge_stiffness(angles)).sites.sum() / 2
        + (overruns & stop_stiffness(overruns)).sites.sum() / 2
    )
    ball_motors = (balls * motors).batch()                                   # [...] Motor+
    world_surfaces = ball_motors >> surfaces(ball_motors << Point)           # [...] Plane+ <- Point-
    bottoms = contact.lowest_point(world_surfaces, ground.dual()).field()    # [...] Point-[n_balls]
    depths = (-(ground & bottoms)).clip(0, np.inf)                           # [...] Scalar+[n_balls]
    # Dualizing downward gravity gives the upward level plane scaled by gravitational field strength.
    gravitational = (gravity.dual() & (motors >> mass_points)).sites.sum()   # [...] Scalar+
    return kinetic + gravitational + spring_energy + (contact_stiffness * depths * depths).sites.sum() / 2


energies = stack([
    mechanical_energy(
        tree,
        home_axes,
        inertias,
        mass_points,
        gravity,
        hinge_stiffness,
        low_stops,
        high_stops,
        stop_stiffness,
        ground,
        surfaces,
        contact_stiffness,
        state,
    )
    for state in states
])                                                                           # [n_frames, 1] Scalar+

In [ ]:
# checks
motor, rate, angles, speeds = states[-1]
motors = forward_fn(motor, angles)
weights = gravity & (motors >> mass_points)                                  # [1] Forque-[V]
forque, forces = inverse_dynamics(
    tree, home_axes, inertias, motors, rate, speeds, rate, speeds, weights
)
loaded = weights + tree[0].adjugate() * forque[..., None].field()            # [1] Forque-[V]
recovered_accelerations = forward_dynamics(
    tree, home_axes, inertias, motors, rate, speeds, forces, loaded
)[1]
assert ((recovered_accelerations - speeds).norm() < 1e-8).all()

n_joints = len(scenarios.joint_names)
_, _, joint_block = mass_matrix(tree, home_axes, inertias, motors)
unit_accelerations = mv.scalar(np.eye(n_joints)[:, None, :, None]).field()   # [n_joints, 1] Scalar+[E]
_, inverse_columns = inverse_dynamics(
    tree, home_axes, inertias, motors, rate * 0, unit_accelerations * 0, rate * 0, unit_accelerations, weights * 0
)
mass_entries = (unit_accelerations[None, :] & joint_block(unit_accelerations)[:, None]).sites.sum()   # [n_joints, n_joints, 1] Force-
transposed_entries = (unit_accelerations[:, None] & joint_block(unit_accelerations)[None, :]).sites.sum()   # [n_joints, n_joints, 1] Force-
inverse_entries = (unit_accelerations[None, :] & inverse_columns[:, None]).sites.sum()                # [n_joints, n_joints, 1] Force-
assert ((mass_entries - inverse_entries).antinorm() < 1e-10).all()
assert ((mass_entries - transposed_entries).antinorm() < 1e-10).all()

assert (((energies[1:] - energies[:-1]) / energies[0].abs()) < 0.03).all()
assert ((energies[-1] / energies[0]) < 0.25).all()